In [ ]:
import torch
import numpy as np

import training_models
from rejector import Rejector
import training
from ensemble import TorchRejectionEnsemble
from DMTimeShardDataset import DMTimeShardDataset

from torch.utils.data import DataLoader, TensorDataset, Subset

import math

In [ ]:
device = "cuda"


r= training_models.models_htable["DM_time_binary_classificator_241002_3_dropout"](256, use_freq_time=True, device=device).to(device)
rejector = Rejector(r, device)

small_weights = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints/ch_point_DM_time_binary_classificator_241002_3_dropout_256/prot-015-0.980-0.985.pth"

small_model = training_models.models_htable["DM_time_binary_classificator_241002_3_dropout"](256, use_freq_time=False, device=device).to(device)
small_model.load_state_dict(torch.load(small_weights, map_location=device)["model_state_dict"])


big_weights = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints/ch_point_DM_time_binary_classificator_resnet18_dropout_256/prot-014-0.972-0.977.pth"

big_model = training_models.models_htable['DM_time_binary_classificator_resnet18_dropout'](256, use_freq_time=True, device=device).to(device)
big_model.load_state_dict(torch.load(big_weights, map_location=device)["model_state_dict"])


rejection_ensemble = TorchRejectionEnsemble(small_model, big_model, p=0.8, rejector=rejector, calibration=False)

In [ ]:
dataset_cfg = {
        "output_dir": "/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs",
        "prefix": "B0531+21_59000_48386",
    }

full_train_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="train")
full_train_dataset.labels = training.label_encoding(full_train_dataset.labels.astype(object))


#val_fraction = 0.1111
#if not 0 < val_fraction < 1:
#    raise ValueError("'val_fraction' must be between 0 and 1.")

#num_train_samples = len(full_train_dataset)
#if num_train_samples < 2:
#    raise ValueError("Need at least 2 training samples to create a validation split.")

#split_idx_val = math.floor(num_train_samples * (1 - val_fraction))
#split_idx_val = min(max(split_idx_val, 1), num_train_samples - 1)

#train_indices = range(0, split_idx_val)
#val_indices = range(split_idx_val, num_train_samples)

#train_dataset = Subset(full_train_dataset, train_indices)
#val_dataset = Subset(full_train_dataset, val_indices)

#test_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="test")
#test_dataset.labels = training.label_encoding(test_dataset.labels.astype(object))

train_dataset = full_train_dataset

test_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="test")
test_dataset.labels = training.label_encoding(full_train_dataset.labels.astype(object))

train_loader = DataLoader(train_dataset,
                          batch_size=256,
                          shuffle=False,
                          num_workers=8)
#val_loader = DataLoader(val_dataset,
#                          batch_size=256,
#                          shuffle=False,
#                          num_workers=8)
test_loader = DataLoader(test_dataset,
                        batch_size=256,
                        shuffle=False,
                        num_workers=8)

print("Train Loader length: ", len(train_loader))
#print("Val Loader length: ", len(val_loader))
print("Test Loader length: ", len(test_loader))
print("train dataset length: ", len(train_dataset))
#print("val dataset length: ", len(val_dataset))
print("test dataset length: ", len(test_dataset))

rejection_ensemble.fit(train_loader, test_loader)